# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sehreen-Atta/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [9]:
import os
from pathlib import Path
import duckdb

env_path = Path(".env")
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, _, value = line.partition("=")
            os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("Set HF_TOKEN in .env")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":       f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":       f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d":    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

One row = one content item's daily performance for one client, on 
one report_date, from fact_content_daily_performance (grain: 
client_hash_id × content_hash_id × report_date). I'll develop on a 
mid-panel month, month=2026-03, and treat the final month (June 
2026, also available separately as fact_content_daily_performance_sample) 
as a sealed test month, since it's the natural outcome window for 
any past→future label.

In [10]:
march_src = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

result = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {march_src}
""").df()
print(result)


   row_count   min_date   max_date
0    9841378 2026-03-01 2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features (predictive signals): gsc_impressions, gsc_clicks, 
gsc_avg_position, ga4_sessions, ga4_engaged_sessions, ga4_pageviews, 
scroll_events, word_count, char_count, search_volume, competition, 
content_type — these describe content and its recent performance.

Label / proxy: none defined yet at the daily grain — a downstream 
label (e.g. "needs refresh") would come from an OBSERVED FUTURE 
outcome (a decline across a following month), not from a 
hand-written rule. This must never be built from the same month 
used as features, or it becomes leakage.

Context (identifiers, not features): client_hash_id, content_hash_id, 
report_date, month — needed to join/group rows but not fed into a 
model directly, since raw IDs carry no generalizable signal.

Excluded: individual AI-channel breakdowns (ai_chatgpt, ai_perplexity, 
ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other) — excluded 
because each is very sparse per row; I'll only use their sum 
(sessions_ai) as a feature instead. Also excluding is_deleted rows 
entirely (filtered out, not used as a feature) since deleted 
content isn't a valid decision target.

In [11]:
# Confirm the excluded AI-channel columns are indeed sparse (supports the "why")
sparsity = con.sql(f"""
    SELECT 
        AVG(CASE WHEN ai_chatgpt > 0 THEN 1 ELSE 0 END) AS pct_nonzero_chatgpt,
        AVG(CASE WHEN ai_perplexity > 0 THEN 1 ELSE 0 END) AS pct_nonzero_perplexity,
        AVG(CASE WHEN sessions_ai > 0 THEN 1 ELSE 0 END) AS pct_nonzero_sessions_ai
    FROM {march_src}
""").df()
print(sparsity)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   pct_nonzero_chatgpt  pct_nonzero_perplexity  pct_nonzero_sessions_ai
0             0.000323                0.000066                 0.000562


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three checks below: (1) grain — confirming one row truly equals one 
client × content × date combination, (2) row count and date span 
for the March 2026 slice, (3) availability — how many rows survive 
when both GSC and GA4 data are actually present.

In [12]:
# 1. Grain check — total rows should equal distinct (client, content, date) combos
grain_check = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS distinct_combos
    FROM {march_src}
""").df()
print("Grain check:")
print(grain_check)

# 2. Row count and date span (already computed in Section 1, repeated here for the record)
span_check = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {march_src}
""").df()
print("\nRow count and date span:")
print(span_check)

# 3. Availability — rows where both GSC and GA4 data are actually available
availability_check = con.sql(f"""
    SELECT COUNT(*) AS rows_with_both_sources
    FROM {march_src}
    WHERE gsc_data_available IS TRUE AND ga4_data_available IS TRUE
""").df()
print("\nRows with both GSC and GA4 data available:")
print(availability_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check:
   total_rows  distinct_combos
0     9841378          9841378

Row count and date span:
   row_count   min_date   max_date
0    9841378 2026-03-01 2026-03-31

Rows with both GSC and GA4 data available:
   rows_with_both_sources
0                  364347


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice can never tell me about content performance outside the 
client base present in this warehouse, and it can't distinguish 
correlation from causation in traffic changes (e.g., a ranking drop 
might stem from a Google algorithm update, not content quality).

Additionally: not every client has both GSC and GA4 connected 
(client_has_gsc / client_has_ga4 flags vary), so some rows are 
GSC-only or GA4-only — comparing "engagement" across clients 
inconsistently would be misleading without checking these flags 
first. AI-referral traffic (sessions_ai, and the individual ai_* 
columns) is a newer signal and likely sparse/undercounted in 
earlier months of this ~17-month window, so trend comparisons 
involving AI traffic should be treated cautiously, especially 
earlier in the date range.

In [13]:
# Confirm the GSC/GA4 availability imbalance mentioned above
source_mix = con.sql(f"""
    SELECT 
        AVG(CASE WHEN client_has_gsc THEN 1 ELSE 0 END) AS pct_has_gsc,
        AVG(CASE WHEN client_has_ga4 THEN 1 ELSE 0 END) AS pct_has_ga4,
        AVG(CASE WHEN client_has_gsc AND client_has_ga4 THEN 1 ELSE 0 END) AS pct_has_both
    FROM {march_src}
""").df()
print(source_mix)


   pct_has_gsc  pct_has_ga4  pct_has_both
0          1.0      0.69326       0.69326


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.